# Extracción de stock histórico mensual (2 años) — OData SAP
**Fuente:** `ZZ1_BI_MM_STOCK_Q_CDS`  
**Requiere:** archivo `.env` en el mismo directorio con `SAP_USER` y `SAP_PASSWORD`, y estar en la red/VPN de Molinos.

In [ ]:
import requests
from requests.auth import HTTPBasicAuth
from dotenv import dotenv_values
from datetime import date
import calendar
import pandas as pd
import time

BASE_URL = "https://fiori.mrp.com.ar/sap/opu/odata/sap/ZZ1_BI_MM_STOCK_Q_CDS/ZZ1_BI_MM_STOCK_Q"

SELECT_FIELDS = [
    "Material",
    "Plant",
    "MatlDocLatestPostgDate",
    "MatlWrhsStkQtyInMatlBaseUnit",
    "MaterialBaseUnit",
]

PAGE_SIZE = 5000
MESES_HISTORIA = 24

config = dotenv_values(".env")
AUTH = HTTPBasicAuth(config["SAP_USER"], config["SAP_PASSWORD"])

print("Config cargada. Usuario:", config.get("SAP_USER"))

In [ ]:
def generar_fin_de_mes(n_meses: int) -> list:
    hoy = date.today()
    fechas = []
    y, m = hoy.year, hoy.month
    for _ in range(n_meses):
        m -= 1
        if m == 0:
            m = 12
            y -= 1
        ultimo_dia = calendar.monthrange(y, m)[1]
        fechas.append(f"{y}{m:02d}{ultimo_dia:02d}")
    return list(reversed(fechas))

fechas = generar_fin_de_mes(MESES_HISTORIA)
print(f"Fechas a extraer ({len(fechas)}): {fechas[0]} ... {fechas[-1]}")

## Paso 1 — Test con una sola fecha
Correr esta celda primero para validar credenciales, formato de fecha y volumen de datos antes de extraer todo.

In [ ]:
def test_una_fecha(fecha: str):
    params = {
        "$select": ",".join(SELECT_FIELDS),
        "$filter": f"MatlDocLatestPostgDate eq '{fecha}'",
        "$format": "json",
        "$top": "10",
    }
    resp = requests.get(BASE_URL, params=params, auth=AUTH,
                        headers={"Accept": "application/json"})
    print("Status:", resp.status_code)
    print("URL:", resp.url)
    print(resp.text[:2000])
    return resp

# Testa con la fecha de fin de mes más reciente
test_una_fecha(fechas[-1])

### Si el filtro de fecha falla (0 resultados o error), probar con formato datetime:
```python
# Reemplazar el filtro por:
"$filter": f"MatlDocLatestPostgDate eq datetime'{fecha[:4]}-{fecha[4:6]}-{fecha[6:8]}T00:00:00'"
```

## Paso 2 — Extracción completa (24 meses)
Solo correr una vez validado el test de arriba.

In [ ]:
def extraer_fecha(fecha: str) -> pd.DataFrame:
    filas = []
    skip = 0
    while True:
        params = {
            "$select": ",".join(SELECT_FIELDS),
            "$filter": f"MatlDocLatestPostgDate eq '{fecha}'",
            "$format": "json",
            "$top": str(PAGE_SIZE),
            "$skip": str(skip),
        }
        resp = requests.get(BASE_URL, params=params, auth=AUTH,
                            headers={"Accept": "application/json"})
        resp.raise_for_status()
        data = resp.json()["d"]["results"]
        if not data:
            break
        filas.extend(data)
        if len(data) < PAGE_SIZE:
            break
        skip += PAGE_SIZE
        time.sleep(0.2)
    df = pd.DataFrame(filas)
    if not df.empty:
        df = df.drop(columns=["__metadata"], errors="ignore")
    return df


def extraer_historico(fechas: list) -> pd.DataFrame:
    dfs = []
    for fecha in fechas:
        print(f"Extrayendo {fecha} ...", end=" ")
        df_fecha = extraer_fecha(fecha)
        print(f"{len(df_fecha)} filas")
        dfs.append(df_fecha)
    return pd.concat(dfs, ignore_index=True)


df_historico = extraer_historico(fechas)

df_historico = df_historico.rename(columns={
    "Material": "material",
    "Plant": "centro",
    "MatlDocLatestPostgDate": "fecha",
    "MatlWrhsStkQtyInMatlBaseUnit": "cantidad_stock",
    "MaterialBaseUnit": "unidad",
})
df_historico["cantidad_stock"] = pd.to_numeric(df_historico["cantidad_stock"], errors="coerce")

out_path = "stock_historico_mensual.csv"
df_historico.to_csv(out_path, index=False)
print(f"\nListo. {len(df_historico)} filas guardadas en {out_path}")
df_historico.head()